In [1]:
!pip install pyoxigraph
!pip install tqdm

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [2]:
data_path = '../../data/raw/dblp.ttl'

In [3]:
from pyoxigraph import Store, RdfFormat
from tqdm import tqdm
import json

# Persistent database stored on disk
store = Store("../../data/raw/dblp_store")

In [4]:


query = """
PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT DISTINCT ?venue ?type ?name
WHERE {
    ?type rdfs:subClassOf+ dblp:Stream .

    ?venue rdf:type ?type .

    OPTIONAL {
        ?venue dblp:primaryStreamTitle ?name
    }
}
LIMIT 10
"""

results = store.query(query)

conferences_data = {}
journal_data = {}

for row in tqdm(results):
    data = {
        "name": row['name'].value
    }

    if row['type'].value == "https://dblp.org/rdf/schema#Conference":
        conference_id = row['venue'].value.replace('https://dblp.org/streams/conf/', '')
        assert conference_id not in conferences_data, f"Duplicate conference venue found: {row['venue'].value}"
        conferences_data[conference_id] = data
    
    elif row['type'].value == "https://dblp.org/rdf/schema#Journal":
        journal_id = row['venue'].value.replace('https://dblp.org/streams/journals/', '')
        assert journal_id not in journal_data, f"Duplicate journal venue found: {row['venue'].value}"
        journal_data[journal_id] = data

print("5 Conferences examples:")
for i, (venue, data) in enumerate(conferences_data.items()):
    if i >= 5:
        break
    print(f"{venue}: {data}")

print('\n')

print("5 Journals examples:")
for i, (venue, data) in enumerate(journal_data.items()):
    if i >= 5:
        break
    print(f"{venue}: {data}")

10it [00:00, 47.06it/s]

5 Conferences examples:


5 Journals examples:


In [5]:
count_query = """
PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT (COUNT(DISTINCT ?venue) AS ?total)
WHERE {
    ?venue rdf:type dblp:Stream .
}
"""

actual_total = int(list(store.query(count_query))[0]["total"].value)

print(f"Actual query rows: {actual_total}")


Actual query rows: 9347


In [6]:
query = """
PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?institution ?name ?country ?state ?city
WHERE {
    ?institution rdf:type dblp:Institution .

    OPTIONAL {
    ?institution dblp:institutionName ?institutionName .
    }
    OPTIONAL {
        ?institution dblp:primaryInstitutionName ?primaryName .
    }
    BIND(COALESCE(?primaryName, ?institutionName) AS ?name)

    OPTIONAL {
        ?institution dblp:hasLocation ?location .

        OPTIONAL {
            ?location dblp:countryName ?country .
        }

        OPTIONAL {
            ?location dblp:stateName ?state .
        }

        OPTIONAL {
            ?location dblp:cityName ?city .
        }
    }
}
LIMIT 1000
"""
with open('../../data/utils/corrected_institution_duplicates.json', 'r') as f:
    corrected_institution_duplicates = json.load(f)

institutions_data = corrected_institution_duplicates.copy()
results = store.query(query)

for row in tqdm(results):

    institution_id = row['institution'].value.replace("https://dblp.org/inst/", "")
    data = {
        "name": row['name'].value
    }

    if row['country']:
        data["country"] = row['country'].value
    if row['state']:
        data["state"] = row['state'].value
    if row['city']:
        data["city"] = row['city'].value

    if not institution_id in institutions_data:
        institutions_data[institution_id] = data
    else:
        if len(data) > len(institutions_data[institution_id]):
            institutions_data[institution_id] = data

        if len(data) == len(institutions_data[institution_id]) and  data != institutions_data[institution_id] and institution_id not in corrected_institution_duplicates:
            raise ValueError(f"Duplicate institution found: ID {institution_id} with data: {data} and {institutions_data[institution_id]}. Run src/scripts/utils/get_institutions_duplicates.py to get the list of duplicates and correct them.")


print(f"Number of institutions after correcting duplicates: {len(institutions_data)}")
print('5 Institutions examples:')
for i, (institution_id, data) in enumerate(institutions_data.items()):
    if i >= 5:
        break
    print(f"{institution_id}: {data}")

0it [00:00, ?it/s]

212it [00:08, 23.76it/s]


KeyboardInterrupt: 

In [ ]:
query = """
PREFIX dblp: <https://dblp.org/rdf/schema#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>

SELECT DISTINCT ?publication ?stream ?institution ?year ?type
WHERE {
    ?stream rdf:type ?type .

    FILTER (?type != dblp:Stream)

    ?publication dblp:yearOfPublication ?year ;
                 dblp:publishedInStream ?stream ;
                 dblp:hasSignature ?signature .

    ?signature dblp:signatureInstitution ?institution .
}
LIMIT 10
"""

results = store.query(query)

conferences_data = {}
journal_data = {}

for row in tqdm(results):

    institution_id = row['institution'].value.replace("https://dblp.org/inst/", "")
    year = row['year'].value
    if row['type'].value == "https://dblp.org/rdf/schema#Conference":
        conference_id = row['stream'].value.replace('https://dblp.org/streams/conf/', '')
        if conference_id not in conferences_data:
            conferences_data[conference_id] = {}
        if year not in conferences_data[conference_id]:
            conferences_data[conference_id][year] = {}
        conferences_data[conference_id][year][institution_id] = conferences_data[conference_id][year].get(institution_id, 0) + 1
    elif row['type'].value == "https://dblp.org/rdf/schema#Journal":
        journal_id = row['stream'].value.replace('https://dblp.org/streams/journals/', '')
        if journal_id not in journal_data:
            journal_data[journal_id] = {}
        if year not in journal_data[journal_id]:
            journal_data[journal_id][year] = {}
        journal_data[journal_id][year][institution_id] = journal_data[journal_id][year].get(institution_id, 0) + 1

    print(row['type'].value)
    print(f"Processed publication {row['publication'].value} in stream {row['stream'].value} for institution {institution_id} in year {year}")
    

print(conferences_data)
print(journal_data)

0it [00:00, ?it/s]

3it [00:00, 27.84it/s]

https://dblp.org/rdf/schema#Journal
Processed publication https://dblp.org/rec/journals/jsyml/Rosser36a in stream https://dblp.org/streams/journals/jsyml for institution 7/285 in year 1936
https://dblp.org/rdf/schema#Journal
Processed publication https://dblp.org/rec/journals/jsyml/Lewis36 in stream https://dblp.org/streams/journals/jsyml for institution 7/285 in year 1936
https://dblp.org/rdf/schema#Journal
Processed publication https://dblp.org/rec/journals/jsyml/Quine36 in stream https://dblp.org/streams/journals/jsyml for institution 7/285 in year 1936
https://dblp.org/rdf/schema#Journal
Processed publication https://dblp.org/rec/journals/jsyml/Emch36 in stream https://dblp.org/streams/journals/jsyml for institution 12/065 in year 1936
https://dblp.org/rdf/schema#Journal
Processed publication https://dblp.org/rec/journals/jsyml/Rosser36 in stream https://dblp.org/streams/journals/jsyml for institution 7/285 in year 1936
https://dblp.org/rdf/schema#Journal
Processed publication http

10it [00:00, 22.45it/s]

https://dblp.org/rdf/schema#Journal
Processed publication https://dblp.org/rec/journals/jsyml/Rosser37 in stream https://dblp.org/streams/journals/jsyml for institution 5/357 in year 1937
https://dblp.org/rdf/schema#Journal
Processed publication https://dblp.org/rec/journals/jsyml/Robinson37 in stream https://dblp.org/streams/journals/jsyml for institution 2/622 in year 1937
https://dblp.org/rdf/schema#Journal
Processed publication https://dblp.org/rec/journals/jsyml/Robinson37 in stream https://dblp.org/streams/journals/jsyml for institution 1/019 in year 1937
https://dblp.org/rdf/schema#Journal
Processed publication https://dblp.org/rec/journals/x/Turing37 in stream https://dblp.org/streams/journals/proclms for institution 1/051 in year 1937
{}
{'jsyml': {'1936': {'7/285': 4, '12/065': 1, '1/100': 1}, '1937': {'5/357': 1, '2/622': 1, '1/019': 1}}, 'proclms': {'1937': {'1/051': 1}}}
